# The Game Engine

This notebook documents the `Board` class, which holds the logic for the whole game engine.
Every later phase (solver, probability engine, simulation runner) interacts with this engine, so getting the conventions right here is important.

### Outline
1. [Board creation](#1-board-creation)
2. [Two grid architecture](#2-two-grid-architecture)
3. [First click safety](#3-first-click-safety)
4. [Reveal & flood fill](#4-reveal--flood-fill)
5. [Game state tracking](#5-game-state-tracking)
6. [Known mine tracking (solver flags)](#6-known-mine-tracking)
7. [Solver-facing queries](#7-solver-facing-queries)
8. [Simulation-only methods](#8-simulation-only-methods)
9. [Visual board printer](#9-visual-board-printer)
10. [How future phases use this engine](#10-how-future-phases-use-this-engine)

## 0. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import random
from src.game.board import Board, HIDDEN, KNOWN_MINE, print_board

## 1. Board creation

To create a board, we need to pass three arguments: `rows`, `cols`, and `mine_count`.
An optional `rng` (a `random.Random` instance) can be passed for reproducibility, which is useful when we start generating boards for our simulation and want to inspect a specific baord more closely.

As already discussed in the problem definition, mines are *not* placed at construction time.
The board is blank until the first `reveal()`

In [2]:
rng = random.Random(42)
board = Board(rows=5, cols=5, mine_count=5, rng=rng)

print(f"Board size: {board.rows}x{board.cols} = {board.total_cells} cells")
print(f"Mines: {board.mine_count}")
print(f"Revealed: {board.revealed_count}")
print(f"Game state: {board.game_state}")
print(f"Mines generated yet? {board._generated}")

Board size: 5x5 = 25 cells
Mines: 5
Revealed: 0
Game state: GameState.ONGOING
Mines generated yet? False


Invalid configurations, such as too many mines or negative baord dimensions, raise a `ValueError`:

In [3]:
import traceback

for kwargs in [
    dict(rows=-1, cols=5, mine_count=5),
    dict(rows=5, cols=5, mine_count=-1),
    dict(rows=5, cols=5, mine_count=26),
]:
    try:
        Board(**kwargs)
    except ValueError as e:
        print(f"Board({kwargs}) -> ValueError: {e}")

Board({'rows': -1, 'cols': 5, 'mine_count': 5}) -> ValueError: rows and cols must be > 0
Board({'rows': 5, 'cols': 5, 'mine_count': -1}) -> ValueError: mine_count must be >= 0
Board({'rows': 5, 'cols': 5, 'mine_count': 26}) -> ValueError: mine_count exceeds total cells


## 2. Two grid architecture

The board maintains two separate grids to track the game:

| Grid | Type | Purpose |
|------|------|---------|
| `_mine_grid` | `list[list[bool]]` | Shows`True` in cells which are mines. Only the engine/simulation should touch this. |
| `_player_grid` | `list[list[int]]` | Represents the game board as it would usually. Encoded as integers (see below). |

### Player grid value encoding

| Value | Meaning |
|-------|---------|
| `-2` (`HIDDEN`) | Cell has not been revealed and is not flagged |
| `-1` (`KNOWN_MINE`) | Cell has been flagged as a known mine|
| `0`–`8` | Cell has been revealed, number shows how many adjacent mines |

This encoding means a single integer conveys everything about a cell's visible state.
Revealed cells are non-negative, hidden/flagged cells are negative.

In [4]:
print("Player grid before any reveal (all HIDDEN = -2):")
for row in range(board.rows):
    print([board.player_value(row, col) for col in range(board.cols)])

Player grid before any reveal (all HIDDEN = -2):
[-2, -2, -2, -2, -2]
[-2, -2, -2, -2, -2]
[-2, -2, -2, -2, -2]
[-2, -2, -2, -2, -2]
[-2, -2, -2, -2, -2]


## 3. First click safety

The first `reveal()` call triggers board generation.
The clicked cell *and all 8 of its neighbors* are guaranteed to be mine-free, since the first cell should always be a zero cell.
This ensures the first click always opens a useful area, like in the normal game.

Mines are placed uniformly at random from the remaining candidate cells.

In [5]:
board = Board(5, 5, mine_count=15, rng=random.Random(42))
first_r, first_c = 2, 2
board.reveal(first_r, first_c)

print(f"Mines generated? {board._generated}")
print(f"\nClicked ({first_r}, {first_c}): is_mine = {board.is_mine(first_r, first_c)}")
print(f"\nSafe zone check — clicked cell + all neighbors should be mine-free:")
for nr, nc in board.neighbors(first_r, first_c):
    assert not board.is_mine(nr, nc), f"Mine at ({nr},{nc})!"
print("All safe — no mines in the safe zone.")

print(f"\nMine grid after generation:")
for row in range(board.rows):
    print(["M" if board.is_mine(row, col) else "." for col in range(board.cols)])

Mines generated? True

Clicked (2, 2): is_mine = False

Safe zone check — clicked cell + all neighbors should be mine-free:
All safe — no mines in the safe zone.

Mine grid after generation:
['M', 'M', 'M', '.', 'M']
['M', '.', '.', '.', 'M']
['M', '.', '.', '.', 'M']
['M', '.', '.', '.', 'M']
['M', 'M', 'M', 'M', 'M']


## 4. Reveal & flood fill

`reveal(row, col)` does the following:
1. If the board hasn't been generated yet, generates it using the first click convention as above.
2. If the cell is a mine, marks the game as `LOST` and returns `False`.
3. If the cell is already revealed or a known mine, returns `True`.
4. Otherwise, reveals the cell and if its adjacency count is 0, performs a **BFS flood fill** that reveals all connected zero-cells and their numbered borders.
5. Checks if all safe cells have been revealed (win condition).

**Convention:** `reveal()` raises `RuntimeError` if called after the game is over (won or lost).

In [6]:
board = Board(8, 8, mine_count=10, rng=random.Random(99))
board.reveal(0, 0)

print("Player grid after first click (flood-fill opened a region):")
print("Legend:  . = hidden (-2);  * = known mine (-1);  numbers = revealed")
for row in range(board.rows):
    line = ""
    for col in range(board.cols):
        v = board.player_value(row, col)
        if v == HIDDEN:
            line += " . "
        elif v == KNOWN_MINE:
            line += " * "
        else:
            line += f" {v} "
    print(line)

print(f"\nRevealed {board.revealed_count}/{board.total_cells - board.mine_count} safe cells")

Player grid after first click (flood-fill opened a region):
Legend:  . = hidden (-2);  * = known mine (-1);  numbers = revealed
 0  0  1  .  .  .  .  . 
 0  0  1  .  .  .  .  . 
 1  1  1  .  .  .  .  . 
 .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  . 
 .  .  .  .  .  .  .  . 

Revealed 9/54 safe cells


Revealing a mine returns `False` and sets game state to `LOST`.
After that, any further `reveal()` call raises `RuntimeError`.

In [7]:
board = Board(5, 5, mine_count=20, rng=random.Random(7))
board.reveal(0, 0)

# Find a mine to reveal
for r in range(5):
    for c in range(5):
        if board.is_mine(r, c) and board.is_hidden(r, c):
            result = board.reveal(r, c)
            print(f"Revealed mine at ({r},{c}): returned {result}")
            print(f"Game state: {board.game_state}")
            try:
                board.reveal(4, 4)
            except RuntimeError as e:
                print(f"Subsequent reveal raises: {e}")
            break
    else:
        continue
    break

Revealed mine at (0,2): returned False
Game state: GameState.LOST
Subsequent reveal raises: Cannot reveal after game is over


## 5. Game state tracking

The board tracks three states via the `GameState` enum:

| State | `game_state` | `won` | `lost` | `game_over` |
|-------|-------------|-------|--------|-------------|
| Ongoing | `ONGOING` | `False` | `False` | `False` |
| Won | `WON` | `True` | `False` | `True` |
| Lost | `LOST` | `False` | `True` | `True` |

Win is detected automatically when `revealed_count == total_cells - mine_count`.

In [8]:
board = Board(3, 3, mine_count=1, rng=random.Random(10))
print(f"Before first click:  state={board.game_state}, game_over={board.game_over}")

board.reveal(0, 0)
print(f"After first click:   state={board.game_state}, revealed={board.revealed_count}")

# Reveal remaining safe cells
for r in range(3):
    for c in range(3):
        if board.is_hidden(r, c) and not board.is_mine(r, c):
            board.reveal(r, c)

print(f"After clearing:      state={board.game_state}, won={board.won}")

Before first click:  state=GameState.ONGOING, game_over=False
After first click:   state=GameState.ONGOING, revealed=6
After clearing:      state=GameState.WON, won=True


## 6. Known mine tracking

The solver uses `mark_known_mine(row, col)` to flag cells it has deduced to be mines.
This writes `KNOWN_MINE` (-1) into the player grid and adds the cell to `_known_mines`.

**Validations (raises on error):**
- `IndexError` if out of bounds
- `ValueError` if cell is not hidden
- `ValueError` if cell is already marked

Once marked, the cell is no longer counted as hidden.

In [9]:
board = Board(5, 5, mine_count=10, rng=random.Random(55))
board.reveal(2, 2)

# Find a hidden cell and mark it
for r in range(5):
    for c in range(5):
        if board.is_hidden(r, c):
            board.mark_known_mine(r, c)
            print(f"Marked ({r},{c}) as known mine.")
            print(f"  player_value: {board.player_value(r, c)} (== KNOWN_MINE: {board.player_value(r, c) == KNOWN_MINE})")
            print(f"  is_known_mine: {board.is_known_mine(r, c)}")
            print(f"  remaining_total_mines: {board.remaining_total_mines()}")
            break
    else:
        continue
    break

Marked (0,0) as known mine.
  player_value: -1 (== KNOWN_MINE: True)
  is_known_mine: True
  remaining_total_mines: 9


## 7. Solver-facing queries

These methods provide the deduction engine with everything it needs.
It is important that none of them use `is_mine()`, since the solver can only use deduction from the board information to make guaranteed moves.

### Frontier
The **frontier** is the collection of all frontier cells, as described in the problem definition.
These are the only cells the solver needs to reason about.

In [10]:
board = Board(8, 8, mine_count=10, rng=random.Random(77))
board.reveal(4, 4)

front = board.frontier()
print(f"Frontier cells ({len(front)}): {front}")

print("\nBoard with frontier marked (F = frontier, . = hidden, numbers = revealed):")
front_set = set(front)
for row in range(board.rows):
    line = ""
    for col in range(board.cols):
        v = board.player_value(row, col)
        if (row, col) in front_set:
            line += " F "
        elif v == HIDDEN:
            line += " . "
        elif v == KNOWN_MINE:
            line += " * "
        else:
            line += f" {v} "
    print(line)

Frontier cells (13): [(0, 4), (0, 5), (1, 5), (2, 5), (2, 6), (3, 6), (3, 7), (4, 7), (5, 2), (5, 7), (6, 7), (7, 6), (7, 7)]

Board with frontier marked (F = frontier, . = hidden, numbers = revealed):
 0  0  0  1  F  F  .  . 
 0  0  0  1  2  F  .  . 
 0  0  0  0  1  F  F  . 
 0  0  0  0  1  2  F  F 
 0  1  1  1  0  1  4  F 
 0  1  F  1  0  0  3  F 
 0  1  1  1  0  1  3  F 
 0  0  0  0  0  1  F  F 


### Constraint queries

For each revealed cell, the solver can query:
- `adj_remaining_mines(r, c)`: how many mines are still unaccounted for among its neighbors
- `count_unrevealed_neighbors(r, c)`: how many hidden neighbors remain
- `unrevealed_neighbors(r, c)`: the list of hidden neighbor cells


In [11]:
board = Board(6, 6, mine_count=5, rng=random.Random(33))
board.reveal(0, 0)

# Walk through every revealed cell and show its constraint state
print("Constraint state for each revealed cell:")
print(f"  {'Cell':<10} {'adj_mines':>9} {'remaining':>9} {'unrev':>5}  verdict")
print(f"  {'-'*55}")
for r in range(6):
    for c in range(6):
        if board.is_revealed(r, c):
            adj = board.adj_mines_num(r, c)
            rem = board.adj_remaining_mines(r, c)
            unr = board.count_unrevealed_neighbors(r, c)
            if adj == 0:
                verdict = "(no hidden neighbors)"
            elif rem == 0:
                verdict = "-> ALL neighbors safe"
            elif rem == unr and unr > 0:
                verdict = "-> ALL neighbors are mines"
            else:
                verdict = f"-> {rem} mines among {unr} unknowns"
            print(f"  ({r},{c})      {adj:>9} {rem:>9} {unr:>5}  {verdict}")

Constraint state for each revealed cell:
  Cell       adj_mines remaining unrev  verdict
  -------------------------------------------------------
  (0,0)              0         0     0  (no hidden neighbors)
  (0,1)              1         1     1  -> ALL neighbors are mines
  (1,0)              0         0     0  (no hidden neighbors)
  (1,1)              1         1     1  -> ALL neighbors are mines
  (1,2)              2         2     4  -> 2 mines among 4 unknowns
  (2,0)              0         0     0  (no hidden neighbors)
  (2,1)              0         0     0  (no hidden neighbors)
  (2,2)              1         1     3  -> 1 mines among 3 unknowns
  (3,0)              1         1     2  -> 1 mines among 2 unknowns
  (3,1)              1         1     3  -> 1 mines among 3 unknowns
  (3,2)              2         2     5  -> 2 mines among 5 unknowns


### Remaining total mines

`remaining_total_mines()` returns the global count of mines which haven't been marked yet.
It decreases as the solver marks known mines.

In [12]:
print(f"Total mines on board: {board.mine_count}")
print(f"Known mines marked:   {len(board._known_mines)}")
print(f"Remaining to find:    {board.remaining_total_mines()}")

Total mines on board: 5
Known mines marked:   0
Remaining to find:    5


## 8. Info methods

These methods are for accessing raw information about a cell.

| Method | Purpose |
|--------|---------|
| `is_mine(r, c)` | Check if cell is a mine - **NOT USED BY SOLVER** |
| `is_revealed(r, c)` | Check if cell has been revealed |
| `is_hidden(r, c)` | Check if cell is still hidden |
| `player_value(r, c)` | Raw integer value from the player grid |
| `neighbors(r, c)` | List of adjacent cell coordinates |
| `adj_mines_num(r, c)` | True adjacent mine count (raises if not revealed) |

## 9. Board printer

This method will be used in future notebooks to make a neat representation of board which can help with visualization and debugging.

In [13]:
board = Board(30, 30, mine_count=270, rng=random.Random(123))
board.reveal(15, 15)

print("Player view:")
print_board(board)

print("\nWith mines revealed (for verification):")
print_board(board, show_mines=True)

Player view:
    0  1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29
    ------------------------------------------------------------------------------------------
 0 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 1 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 2 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 3 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 4 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 5 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 6 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 7 | .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . 
 8 | .  .  .  .  . 

## 10. Summary and next phase

In this notebook, we have succesfully documented our implementation of the Minesweeper game engine, how our defined conventions are used, what functionality it has and how these functions will be used in the future.

The next phase of the project is writing our deduction solver, which will complete all guaranteed moves until it reaches a stale point where the next move must be a guess.